# Mining Energy ETL
Run one code cell at a time. Setup first, then upload the two CSV inputs, then run Extract, Transform and Load.

In [ ]:
import csv
import re
from pathlib import Path

CATALOG = "dbw_sales_iman01"  # Change only if your workspace uses another catalog.
SCHEMA = "iman_energy_etl_lab"
VOLUME = "incoming"
USE_HISTORY = False  # True for the machine learning exercise
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", CATALOG)
prefix = f"`{CATALOG}`.`{SCHEMA}`"
volume = Path(f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {prefix}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {prefix}.`{VOLUME}`")
print(volume)


## Extract
Upload tariffs.csv and meter_readings.csv (or meter_history.csv if USE_HISTORY=True) before running this cell. The files stay in the volume.

In [ ]:
def extract(filename, expected_header):
    with (volume / filename).open(encoding="utf-8-sig", newline="") as stream:
        reader = csv.DictReader(stream)
        if reader.fieldnames != expected_header:
            raise ValueError(f"Unexpected header in {filename}: {reader.fieldnames}")
        rows = list(reader)
        if any(None in row or None in row.values() for row in rows):
            raise ValueError(f"A row has the wrong number of fields in {filename}")
        return rows

readings = extract("meter_history.csv" if USE_HISTORY else "meter_readings.csv", ["reading_id", "date", "site", "energy_kwh"])
tariffs = extract("tariffs.csv", ["site", "cents_per_kwh"])
print(f"Extracted {len(readings)} readings and {len(tariffs)} tariffs")


## Transform
Calculate each reading cost using its site's tariff. Keep rejected records with reasons; count repeated records separately. Monetary values are stored as integer cents.

In [ ]:
from collections import defaultdict
from datetime import date

METER_FIELDS = ("reading_id", "date", "site", "energy_kwh")
RATE_FIELDS = ("site", "cents_per_kwh")

def transform(readings, tariffs):
    rates = {}
    for row in tariffs:
        site = row["site"].strip().upper()
        rate = int(row["cents_per_kwh"])
        if not site or rate <= 0 or site in rates:
            raise ValueError("Tariffs must have unique sites and positive rates")
        rates[site] = rate

    seen, clean, rejected = {}, [], []
    duplicates = 0
    daily = defaultdict(lambda: [0, 0])
    for raw in readings:
        key = tuple(raw[k].strip() for k in METER_FIELDS)
        reading_id, day, site, kwh_text = key
        site = site.upper()
        if reading_id and reading_id in seen:
            if key != seen[reading_id]:
                raise ValueError(f"Conflicting rows for {reading_id}")
            duplicates += 1
            continue
        if reading_id:
            seen[reading_id] = key

        try:
            if not reading_id:
                raise ValueError("Missing reading ID")
            if date.fromisoformat(day).isoformat() != day:
                raise ValueError("Use ISO date YYYY-MM-DD")
            try:
                kwh = int(kwh_text)
            except ValueError:
                raise ValueError("Energy must be a whole number") from None
            if kwh <= 0:
                raise ValueError("Energy must be positive")
            if site not in rates:
                raise ValueError("No tariff for site")
        except ValueError as exc:
            rejected.append((*key, str(exc)))
            continue

        rate = rates[site]
        cost = kwh * rate
        clean.append((reading_id, day, site, kwh, rate, cost))
        daily[day, site][0] += kwh
        daily[day, site][1] += cost

    summary = [(day, site, *amounts) for (day, site), amounts in sorted(daily.items())]
    assert len(readings) == len(clean) + len(rejected) + duplicates
    return clean, rejected, summary, duplicates


In [ ]:
clean, rejected, daily, duplicates = transform(readings, tariffs)
print(f"Valid: {len(clean)} | Rejected: {len(rejected)} | Duplicates: {duplicates}")
for row in rejected:
    print(row)
if USE_HISTORY:
    assert (len(clean), len(rejected), duplicates, len(daily)) == (224, 1, 1, 112)
else:
    assert (len(clean), len(rejected), duplicates, len(daily)) == (6, 3, 1, 4)
    assert sum(row[3] for row in daily) == 32000


## Load
Write three Delta tables. Re-running replaces the tables in this practice schema; it does not append rows.

In [ ]:
from pyspark.sql import functions as F

def load(rows, schema, name, cast_date=False):
    frame = spark.createDataFrame(rows, schema=schema)
    if cast_date:
        frame = frame.withColumn("date", F.to_date("date"))
    frame.write.format("delta").mode("overwrite").saveAsTable(f"{prefix}.`{name}`")

load(clean, "reading_id STRING, date STRING, site STRING, energy_kwh LONG, cents_per_kwh LONG, cost_cents LONG",
     "silver_energy_readings", cast_date=True)
load(rejected, "reading_id STRING, date STRING, site STRING, energy_kwh STRING, reason STRING",
     "rejected_energy_readings")
load(daily, "date STRING, site STRING, total_kwh LONG, total_cost_cents LONG",
     "gold_daily_energy_cost", cast_date=True)

display(spark.sql(f'''
    SELECT date, site, total_kwh, total_cost_cents / 100.0 AS total_cost_aud
    FROM {prefix}.gold_daily_energy_cost
    ORDER BY date, site
'''))
